# **Project Name**    :-  **Brain Tumor MRI Image Classification**



##### **Project Type**   :- **Classification**
##### **Contribution**    :- **Individual**
##### **Name :- Dewang Bhonde**


# **Project Summary :-**

This project classifies brain MRI images into four supplied categories: Glioma, Meningioma, No Tumor, and Pituitary. The archives contain 1,695 training, 502 validation, and 246 test images (2,443 total), with labels in `_classes.csv`. The notebook loads the actual labels, checks files, explores class distributions and image properties, preprocesses images, trains three convolutional neural network approaches, compares validation performance, and evaluates the selected model on the held-out test set.

A compact CNN trained from scratch is compared with transfer-learning approaches based on MobileNetV2 and EfficientNetB0. Images are resized to a consistent input size and normalized. Accuracy, precision, recall, macro-F1, class-wise reports, and confusion matrices are used to evaluate the models because overall accuracy alone may conceal weak performance on an individual class. The provided split is preserved to reduce data leakage, and the test set is not used for tuning.

This is an educational machine-learning project only. It is not a medical diagnostic system, and predictions must not be used for diagnosis, treatment, or patient-care decisions. Numerical findings are generated when the notebook is executed with the dataset and required libraries available. Add author details, inspect actual outputs, and update the conclusion with measured results before submission.

# **GitHub Link :-**

Provide your GitHub Link here.

# **Problem Statement**


This project aims to develop a deep learning-based solution for classifying brain MRI images into multiple categories according to tumor type. It involves building a custom CNN model from scratch and enhancing performance through transfer learning using pretrained models. The project also includes deploying a user-friendly Streamlit web application to enable real-time tumor type predictions from uploaded MRI images.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Install/import project dependencies
!pip -q install tensorflow scikit-learn seaborn pillow streamlit

import os, zipfile, random, shutil, pathlib, warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support)
from scipy.stats import chi2_contingency
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

### Dataset Loading

In [ ]:

# OPTION A: ZIP uploaded using Colab left sidebar > Files > Upload to session storage
ZIP_PATH="/content/Tumour-20261004T144331Z-1-001.zip"
EXTRACT_TO="/content/brain_tumor_data"
if os.path.isfile(ZIP_PATH):
    os.makedirs(EXTRACT_TO,exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH,"r") as zf:
        print("ZIP sample paths:",zf.namelist()[:12])
        zf.extractall(EXTRACT_TO)
else:
    print("ZIP not found in /content. Upload it in Colab Files sidebar or use Drive option below.")
DATASET_ROOT=os.path.join(EXTRACT_TO,"Tumour")
TRAIN_DIR=os.path.join(DATASET_ROOT,"train")
VAL_DIR=os.path.join(DATASET_ROOT,"valid")
TEST_DIR=os.path.join(DATASET_ROOT,"test")
CLASS_NAMES=["glioma","meningioma","no_tumor","pituitary"]
for label,path in [("train",TRAIN_DIR),("valid",VAL_DIR),("test",TEST_DIR)]:
    print(label,path,os.path.isdir(path))


In [ ]:
import tensorflow as tf
from tensorflow import keras
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

CLASS_NAMES = ["glioma", "meningioma", "no_tumor", "pituitary"]

# Check if the dataset directories exist before loading
if not all(os.path.exists(d) for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]):
    print("[ERROR] Cannot load datasets.")
    print("One or more required dataset directories are missing. Please make sure:")
    print("1. You have uploaded 'Tumour-20261004T144331Z-1-001.zip' to '/content/'.")
    print("2. You have successfully run the previous 'Dataset Loading' cell to extract it.")
else:
    train_ds = keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        class_names=CLASS_NAMES,
        shuffle=True,
        seed=SEED
    )

    val_ds = keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        class_names=CLASS_NAMES,
        shuffle=False
    )

    test_ds = keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
        class_names=CLASS_NAMES,
        shuffle=False
    )

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds = val_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)

    print("Dataset successfully loaded!")
    print("Classes:", CLASS_NAMES)
    print("Number of classes:", len(CLASS_NAMES))

In [ ]:
import os

for split_name, folder in [
    ("Training", TRAIN_DIR),
    ("Validation", VAL_DIR),
    ("Testing", TEST_DIR)
]:
    print(f"\n{split_name} dataset:")
    total = 0

    if not os.path.exists(folder):
        print(f"[WARNING] Folder '{folder}' does not exist. Please check your dataset extraction.")
        continue

    for cls in CLASS_NAMES:
        class_path = os.path.join(folder, cls)
        if os.path.exists(class_path):
            count = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
            ])
        else:
            count = 0
            print(f"[WARNING] Subfolder '{cls}' missing in {split_name}!")
        print(f"{cls}: {count}")
        total += count

    print("Total:", total)

### Dataset First View

In [ ]:
IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp"}
def image_files(folder):
    return sorted(p for p in pathlib.Path(folder).rglob("*")
                  if p.is_file() and p.suffix.lower() in IMG_EXTS)
if not all(os.path.isdir(p) for p in [TRAIN_DIR,VAL_DIR,TEST_DIR]):
    raise FileNotFoundError("Dataset folders not found. Check ZIP_PATH/extraction and expected Tumour/train, valid, test structure.")
classes=sorted([p.name for p in pathlib.Path(TRAIN_DIR).iterdir() if p.is_dir()])
print("Detected classes:",classes)
print("Training sample files:",[str(p) for p in image_files(TRAIN_DIR)[:5]])


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
import os
import pathlib
import pandas as pd
from PIL import Image

split_dirs = {"train": TRAIN_DIR, "valid": VAL_DIR, "test": TEST_DIR}
records = []

for split, folder in split_dirs.items():
    if not os.path.exists(folder):
        print(f"[WARNING] Skipping split '{split}': directory '{folder}' does not exist.")
        continue
    for clsdir in sorted(pathlib.Path(folder).iterdir()):
        if clsdir.is_dir():
            for path in image_files(clsdir):
                try:
                    with Image.open(path) as im:
                        w, h = im.size
                        mode = im.mode
                    records.append({
                        "image_path": str(path),
                        "split": split,
                        "label": clsdir.name,
                        "width": w,
                        "height": h,
                        "mode": mode,
                        "file_ext": path.suffix.lower()
                    })
                except Exception as e:
                    print("Unreadable image skipped:", path, e)

image_df = pd.DataFrame(records)
if not image_df.empty:
    display(image_df.head())
else:
    print("[WARNING] No images successfully loaded into image_df.")

### Dataset Information

In [ ]:
# Dataset Info
print("Rows (images):",len(image_df))
print("Columns (metadata fields):",len(image_df.columns))
print("Shape:",image_df.shape)

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Duplicate paths:",image_df.image_path.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("Missing values by metadata column:")
display(image_df.isna().sum().to_frame("missing_count"))

### What did you know about your dataset?

The dataset is organized as MRI images in class folders. The inventory cells report the real image counts, file types, and readable image dimensions. The supplied split folders are preserved.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Dataset columns:",image_df.columns.tolist())

In [ ]:
# Dataset Describe
display(image_df[["width","height"]].describe())
display(image_df.groupby(["split","label"]).size().unstack(fill_value=0))

### Variables Description

The variable descriptions are shown above the metadata table. 'label' is the target class; 'image paths' and 'technical metadata' are identifiers/attributes, not clinical features.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in ["split","label","mode","file_ext"]:
    print(col,":",sorted(image_df[col].dropna().unique().tolist()))

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Keep only supported image files and expected labels; image_df already contains readable images.
image_df=image_df[image_df.label.isin(classes)].reset_index(drop=True)
print("Clean metadata shape:",image_df.shape)

### What all manipulations have you done and insights you found?

The workflow builds a metadata table from readable images, records folder-derived labels, checks image modes and dimensions, and standardizes image dimensions during model input. Corrupt/unreadable images are skipped.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### **Chart 1 :- Training Images per Tumor Class**

In [ ]:
counts=image_df[image_df.split=="train"].label.value_counts().reindex(classes,fill_value=0)
plt.figure(figsize=(8,4)); sns.barplot(x=counts.index,y=counts.values)
plt.title("Training images per class"); plt.xlabel("Tumor class"); plt.ylabel("Image count"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

 - Bar chart is selected to clearly compare the number of training images across different tumor classes.




##### 2. What is/are the insight(s) found from the chart?

- It shows the class-wise distribution of training images.
- It helps identify imbalanced or underrepresented tumor classes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps improve model reliability by identifying data imbalance and guiding better data collection and model training.
- Negative: If one class has significantly fewer images, the model may show lower accuracy for that tumor class, negatively affecting reliable medical diagnosis.

#### **Chart 2 :- Image Count by Class and Split**

In [ ]:
ct=image_df.groupby(["label","split"]).size().reset_index(name="count")
plt.figure(figsize=(8,4)); sns.barplot(data=ct,x="label",y="count",hue="split")
plt.title("Image count by class and split"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

- Grouped bar chart is used to compare image counts for each tumor class across train, validation, and test splits.

##### 2. What is/are the insight(s) found from the chart?

- It shows whether the dataset is properly distributed across different classes and splits.
- It helps identify class imbalance or insufficient test/validation data.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Ensures balanced and reliable model evaluation, supporting better medical prediction accuracy.
- Negative: Unequal class distribution can cause biased predictions and poor performance for minority classes, reducing the reliability of the system.

#### **Chart 3 :- Sample MRI Images by Tumor Class**

In [ ]:
fig,ax=plt.subplots(2,min(4,len(classes)),figsize=(14,6)); ax=np.array(ax).reshape(2,-1)
for j,c in enumerate(classes[:ax.shape[1]]):
    fs=image_files(pathlib.Path(TRAIN_DIR)/c)
    for r in range(2):
        if fs:
            ax[r,j].imshow(Image.open(fs[min(r,len(fs)-1)]).convert("RGB")); ax[r,j].set_title(c); ax[r,j].axis("off")
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

- Image grid is selected to visually compare sample MRI images from different tumor classes.

##### 2. What is/are the insight(s) found from the chart?

- It shows the visual differences and similarities among tumor classes.
- It helps identify image quality, variations, and possible noise in the dataset.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps improve data quality and model training, leading to better tumor classification.
- Negative: Poor-quality or highly variable images may cause incorrect predictions, reducing the reliability of the medical system.

#### **Chart 4 :- Image Width Distribution**

In [ ]:
plt.figure(figsize=(7,4)); sns.histplot(data=image_df,x="width",hue="split",bins=25,element="step"); plt.title("Image width distribution"); plt.show()

##### 1. Why did you pick the specific chart?

- Histogram is used to understand the distribution of image widths across the dataset

##### 2. What is/are the insight(s) found from the chart?

- It shows the common image sizes and variations in image width.
- It helps identify whether images have consistent or different resolutions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps ensure proper image preprocessing and resizing, improving model performance.
- Negative: Large variations in image sizes may require extra preprocessing and can reduce model accuracy if not handled properly.

#### **Chart 5 :- Image Height Distribution**

In [ ]:
plt.figure(figsize=(7,4)); sns.histplot(data=image_df,x="height",hue="split",bins=25,element="step"); plt.title("Image height distribution"); plt.show()

##### 1. Why did you pick the specific chart?

- Histogram is used to analyze the distribution of image heights across different dataset splits

##### 2. What is/are the insight(s) found from the chart?

- It shows the common image heights and variations in the dataset.
- It helps identify inconsistent image resolutions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Supports proper image resizing and preprocessing, improving model performance.
- Negative: Highly varied image heights may cause inconsistent model inputs and lower prediction accuracy if not standardized.

#### **Chart 6 :- Image Width vs Height**

In [ ]:
plt.figure(figsize=(7,4)); sns.scatterplot(data=image_df.sample(min(1200,len(image_df)),random_state=SEED),x="width",y="height",hue="split",alpha=.5); plt.title("Image width vs height"); plt.show()

##### 1. Why did you pick the specific chart?

- Scatter plot is used to identify the relationship between image width and height across different dataset splits.

##### 2. What is/are the insight(s) found from the chart?

- It shows whether images have similar or varied dimensions.
- It helps identify outliers and inconsistent image resolutions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps in standardizing image dimensions, improving preprocessing and model performance.
- Negative: Highly varied dimensions or outliers can increase preprocessing requirements and may reduce prediction accuracy.

#### **Chart 7 :- Training Class Proportion**

In [ ]:
(image_df[image_df.split=="train"].label.value_counts(normalize=True).reindex(classes,fill_value=0)*100).plot(kind="pie",autopct="%1.1f%%",figsize=(6,6)); plt.title("Training class proportion"); plt.ylabel(""); plt.show()

##### 1. Why did you pick the specific chart?

- Pie chart is selected to show the percentage contribution of each tumor class in the training dataset.

##### 2. What is/are the insight(s) found from the chart?

- It shows the proportion of images in each tumor class.
- It helps identify class imbalance in the training data.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps maintain balanced model training, improving classification reliability.
- Negative: A highly dominant class can make the model biased toward that class, reducing accuracy for minority tumor classes.

#### **Chart 8 :- Width Distribution by Class**

In [ ]:
plt.figure(figsize=(8,4)); sns.boxplot(data=image_df[image_df.split=="train"],x="label",y="width"); plt.xticks(rotation=20); plt.title("Width distribution by class"); plt.show()

##### 1. Why did you pick the specific chart?

- Box plot is used to compare the image width distribution across tumor classes and detect outliers.

##### 2. What is/are the insight(s) found from the chart?

- It shows the median, spread, and variation of image widths for each class.
- It helps identify unusual image dimensions or outliers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps improve data preprocessing and standardization, supporting better model performance.
- Negative: Significant variation or outliers may cause inconsistent model inputs, potentially reducing prediction accuracy.

#### **Chart 9 :- Height Distribution by Class**

In [ ]:
plt.figure(figsize=(8,4)); sns.boxplot(data=image_df[image_df.split=="train"],x="label",y="height"); plt.xticks(rotation=20); plt.title("Height distribution by class"); plt.show()

##### 1. Why did you pick the specific chart?

- Box plot is used to compare the image height distribution across tumor classes and detect outliers.

##### 2. What is/are the insight(s) found from the chart?

- It shows the median, spread, and variation of image heights for each class.
- It helps identify unusual image dimensions and class-wise differences.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps in image standardization and preprocessing, improving model consistency.
- Negative: Large variations or outliers may lead to inconsistent inputs and lower classification accuracy.

#### **Chart 10 :- Sample RGB Channel Means**

In [ ]:
means=[]
for p in image_files(TRAIN_DIR)[:500]:
    a=np.asarray(Image.open(p).convert("RGB").resize((64,64)),dtype=np.float32)/255
    means.append(a.mean((0,1)))
if means: pd.DataFrame(means,columns=["R","G","B"]).boxplot(); plt.title("Sample RGB channel means"); plt.show()

##### 1. Why did you pick the specific chart?

- Box plot is used to compare the mean values of RGB color channels in sample MRI images.

##### 2. What is/are the insight(s) found from the chart?

- It shows the distribution and variation of R, G, and B channel intensities.
- It helps identify color/intensity inconsistencies in the images.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps improve image normalization and preprocessing, supporting better model performance.
- Negative: Large variations in channel intensity may introduce data inconsistency, potentially reducing prediction accuracy.

#### **Chart 11 :- Training Image File Extensions**

In [ ]:
pd.Series([p.suffix.lower() for p in image_files(TRAIN_DIR)]).value_counts().plot(kind="bar",figsize=(6,3)); plt.title("Training image file extensions"); plt.ylabel("Count"); plt.show()

##### 1. Why did you pick the specific chart?

- Bar chart is used to compare the number of images by file extension in the training dataset.

##### 2. What is/are the insight(s) found from the chart?

- It shows which image formats are used in the dataset.
- It helps identify format consistency and the presence of multiple file types.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps ensure consistent file processing, improving preprocessing efficiency.
- Negative: Multiple or unsupported formats may cause loading/processing errors, affecting model training.

#### **Chart 12 :- Train Minus Test Images by Class**

In [ ]:
piv=image_df.groupby(["label","split"]).size().unstack(fill_value=0).reindex(classes,fill_value=0)
(piv["train"]-piv["test"]).plot(kind="bar",figsize=(8,3)); plt.title("Train minus test images by class"); plt.ylabel("Count difference"); plt.show()

##### 1. Why did you pick the specific chart?

- Bar chart is used to compare the difference between training and testing images for each tumor class.

##### 2. What is/are the insight(s) found from the chart?

- It shows whether each class has more training images than test images.
- It helps identify imbalances between training and testing data.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

- Positive: Helps maintain a proper train-test balance, supporting reliable model evaluation.
- Negative: A very large difference may cause overfitting or unreliable testing results, reducing model generalization.

#### **Chart 13 :-Numeric Metadata Correlation Heatmap**

In [ ]:
numeric=image_df.assign(class_index=image_df.label.map({c:i for i,c in enumerate(classes)}))[["width","height","class_index"]]
plt.figure(figsize=(5,4)); sns.heatmap(numeric.corr(),annot=True,cmap="coolwarm",center=0); plt.title("Numeric metadata correlation"); plt.show()

##### 1. Why did you pick the specific chart?

- Heatmap is used to show the correlation between image width, height, and class index.

##### 2. What is/are the insight(s) found from the chart?

- It shows the strength and direction of relationships between numeric variables.
- It helps identify strong or weak correlations in the dataset.

#### **Chart 14 :- Pairwise Relationship of Image Dimensions Plot**

In [ ]:
sns.pairplot(image_df.sample(min(500,len(image_df)),random_state=SEED),vars=["width","height"],hue="split"); plt.show()

##### 1. Why did you pick the specific chart?

- Pair plot is used to compare the relationship between image width and height across train, validation, and test splits.

##### 2. What is/are the insight(s) found from the chart?

-It shows the distribution and relationship between width and height.
- It helps identify similar patterns, variations, and outliers across dataset splits.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Use class-count and split-distribution plots to define hypotheses. The following tests report p-values; statistical significance is not the same as practical or clinical significance.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- H0: The stated groups/distributions do not differ (or are independent, as applicable).
- H1: A difference or association exists.
- Use the p-value printed by the following test and a preselected significance level such as 0.05.

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import chisquare

# H0: training class frequencies are equal; H1: at least one differs.
counts=image_df[image_df.split=="train"].label.value_counts().reindex(classes,fill_value=0)
stat,p=chisquare(counts.values)
print("Chi-square goodness-of-fit:",stat,"p-value:",p)

##### Which statistical test have you done to obtain P-Value?

- Chi-Square Goodness-of-Fit Test was used to calculate the p-value.
- It checks whether the training class frequencies are significantly different from equal frequencies.

##### Why did you choose the specific statistical test?

- It is suitable for categorical frequency/count data such as tumor classes.
- It helps determine whether the class distribution is balanced or significantly different.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- H0: The stated groups/distributions do not differ (or are independent, as applicable).
- H1: A difference or association exists.
 - Use the p-value printed by the following test and a preselected significance level such as 0.05.

#### 2. Perform an appropriate statistical test.

In [ ]:
# H0: class and split are independent; H1: class distribution differs across splits.
ct=pd.crosstab(image_df.label,image_df.split)
stat,p,dof,expected=chi2_contingency(ct)
print("Chi-square independence:",stat,"p-value:",p,"degrees freedom:",dof)

##### Which statistical test have you done to obtain P-Value?

- Chi-Square Test of Independence was used to calculate the p-value.
- It tests whether class and dataset split (train/validation/test) are independent.

##### Why did you choose the specific statistical test?

- It is suitable for categorical data represented as frequency counts.
- It checks whether class distribution differs significantly across the dataset splits.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- H0: The stated groups/distributions do not differ (or are independent, as applicable).
- H1: A difference or association exists.
 - Use the p-value printed by the following test and a preselected significance level such as 0.05.

#### 2. Perform an appropriate statistical test.

In [ ]:
# H0: width distributions are same for train and test; H1: distributions differ.
from scipy.stats import mannwhitneyu

w_train=image_df.loc[image_df.split=="train","width"]
w_test=image_df.loc[image_df.split=="test","width"]
stat,p=mannwhitneyu(w_train,w_test,alternative="two-sided")
print("Mann-Whitney U:",stat,"p-value:",p)

##### Which statistical test have you done to obtain P-Value?

- **Mann-Whitney U Test** (two-sided) was used to calculate the p-value.
- It compares the distribution of image widths between the 'train' and 'test' dataset splits.

##### Why did you choose the specific statistical test?

- It is a non-parametric test suitable for comparing the distributions of continuous/ordinal values (like image resolutions/widths) across two independent groups.
- Unlike a t-test, it does not assume that the image dimensions follow a normal distribution.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Image data has no tabular null cells. Metadata rows represent successfully opened images.
print("Missing metadata:",image_df.isna().sum().sum())
print("Readable image records:",len(image_df))


#### What all missing value imputation techniques have you used and why did you use those techniques?

- No missing value imputation technique was used, because the dataset contains 0 missing values (null cells).
- The metadata includes only successfully readable image records, so imputation was not required.

### 2. Handling Outliers

In [ ]:
# Inspect unusual image sizes; do not remove based only on statistical outlier rules.
display(image_df.sort_values(["width","height"])[["image_path","label","width","height"]].head())
display(image_df.sort_values(["width","height"])[["image_path","label","width","height"]].tail())


##### What all outlier treatment techniques have you used and why did you use those techniques?

- No outlier removal technique was used, because unusual image sizes may still contain valid medical information.
- Instead, image dimensions were inspected and sorted to identify unusual sizes without removing them incorrectly.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
class_to_idx={c:i for i,c in enumerate(classes)}
idx_to_class={i:c for c,i in class_to_idx.items()}
print(class_to_idx)


#### What all categorical encoding techniques have you used & why did you use those techniques?

- Label Encoding was used to convert tumor class names into numeric labels (0, 1, 2, 3).
- It was used because ML models require numerical class representations for classification.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 2. Lower Casing

In [ ]:
# Lower Casing
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

In [ ]:
# Remove White spaces
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 7. Tokenization

In [ ]:
# Tokenization
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 8. Text Normalization

In [ ]:
# Text Normalization

print("TEXT NORMALIZATION:")
print("Not applicable because the dataset does not contain free-text data.")

print("\nIMAGE NORMALIZATION:")
print("Pixel normalization is applied for the MRI images.")

print("For the Custom CNN, pixel values are rescaled from [0, 255] to [0, 1].")

##### Which text normalization technique have you used and why?

- No text normalization technique was used because the dataset does not contain free-text data.
- For the actual MRI image data, pixel normalization/rescaling is used. Pixel values originally range from approximately 0 to 255 and are rescaled to the range 0 to 1 for the Custom CNN. This improves numerical stability and makes model optimization easier.
- Text techniques such as stemming, lemmatization, lowercasing, and stopword removal are not applicable to this image-classification problem.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
print("Not applicable: dataset contains MRI images and folder labels, not free-text NLP records.")

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
print("TEXT VECTORIZATION:")
print("Not applicable because the dataset contains MRI images, not text.")
print("\nIMAGE TENSOR CONVERSION:")
print("MRI images are converted into numerical tensors before being passed to the CNN.")

##### Which text vectorization technique have you used and why?

- Text vectorization was not used because the dataset does not contain textual data.
- Instead, MRI images are converted into numerical tensors that can be processed by convolutional neural networks. Each image is resized to 224 × 224 pixels with 3 RGB channels, producing an input tensor of shape 224 × 224 × 3. A batch of 32 images therefore has a shape of 32 × 224 × 224 × 3.
- For the Custom CNN, pixel values are normalized from 0–255 to 0–1 before training.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
NUM_CLASSES=len(classes)
print("Model classes:",NUM_CLASSES,"Input image size:",IMG_SIZE)

#### 2. Feature Selection

In [ ]:
# No tabular feature selector used: CNN learns spatial filters from MRI image tensors.
print("Feature selection: learned convolutional features; no manual column selection.")


##### What all feature selection methods have you used  and why?

- No manual feature selection method was used.
- The CNN automatically learns important spatial features from MRI images through convolutional layers.

##### Which all features you found important and why?

- Important features include edges, textures, shapes, intensity patterns, and tumor structures.
- These features help the CNN differentiate between different brain tumor classes.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Decode to RGB, resize to common dimensions. Models apply scaling/preprocessing internally.
def load_image(path,label):
    raw=tf.io.read_file(path)
    img=tf.io.decode_image(raw,channels=3,expand_animations=False)
    img=tf.image.resize(img,IMG_SIZE)
    return tf.cast(img,tf.float32),tf.cast(label,tf.int32)
print("Target size:",IMG_SIZE)

- Yes. Images are decoded as RGB and resized to 224×224.
- Augmentation is applied only during training.
- Transfer-learning architectures receive model-appropriate preprocessing.

### 6. Data Scaling

In [ ]:
# Custom CNN uses Rescaling(1./255); MobileNetV2 uses preprocess_input; EfficientNetB0 expects [0,255] float input in Keras.
print("Scaling is handled in each model architecture to avoid double normalization.")


##### Which method have you used to scale you data and why?

- Image normalization/scaling was used, mainly Rescaling by 1/255 for the custom CNN.
- It converts pixel values from 0–255 to 0–1, which improves training stability and model convergence.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

- PCA is not applied because flattening/reducing MRI pixels can discard spatial relationships. CNNs learn compact representations through convolution and pooling.
- The dataset is image-based. The corresponding code cell performs the analysis and prints or visualizes the result. Record the actual output after executing the notebook; no dataset-specific result is assumed.

In [ ]:
# No PCA is applied to images; convolutional networks retain spatial structure and learn compact features.
print("Dimensionality reduction: not applied before CNN.")


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

- The dataset's supplied train/valid/test split is retained. This avoids unnecessarily reshuffling the official splits.
- If patient IDs are available, a patient-level split is preferable to prevent leakage.

### 8. Data Splitting

In [ ]:
# Provided train/valid/test folders are preserved. No extra split is performed.
def make_dataset(folder,shuffle=False):
    return keras.utils.image_dataset_from_directory(
        folder,class_names=classes,label_mode="int",image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,shuffle=shuffle,seed=SEED).prefetch(AUTOTUNE)
train_ds=make_dataset(TRAIN_DIR,True)
val_ds=make_dataset(VAL_DIR,False)
test_ds=make_dataset(TEST_DIR,False)
print("Datasets loaded; keep test_ds for final evaluation only.")


##### What data splitting ratio have you used and why?

- No additional data-splitting ratio was used because the dataset already provides separate Train, Validation, and Test folders.
- The existing split was preserved to ensure fair model training, validation, and final evaluation.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

- The training class-count chart and table show whether the data is imbalanced.
- The code calculates class weights from actual training counts; inspect those outputs before describing imbalance.

In [ ]:
# Handling Imbalanced Dataset (If needed)
counts=image_df[image_df.split=="train"].label.value_counts().reindex(classes,fill_value=0)
total=counts.sum()
class_weights={i:float(total/(len(classes)*counts[c])) for i,c in enumerate(classes) if counts[c]>0}
print("Training counts:",counts.to_dict())
print("Class weights:",class_weights)


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

- Class Weighting technique was used to handle class imbalance.
- Higher weights are assigned to minority classes, helping the CNN give balanced importance to all classes during training.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
EPOCHS = 15
augmentation=keras.Sequential([
    layers.RandomRotation(.04),layers.RandomZoom(.1),
    layers.RandomTranslation(.05,.05),layers.RandomFlip("horizontal")
],name="augmentation")
OUT_DIR="/content/brain_tumor_outputs"; os.makedirs(OUT_DIR,exist_ok=True)
def callbacks(name):
    return [keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(os.path.join(OUT_DIR,name+".keras"),monitor="val_loss",save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.3,patience=2,min_lr=1e-7)]
def build_custom_cnn():
    inp=keras.Input((*IMG_SIZE,3)); x=layers.Rescaling(1./255)(inp); x=augmentation(x)
    for f in [32,64,128]:
        x=layers.Conv2D(f,3,padding="same",use_bias=False)(x); x=layers.BatchNormalization()(x)
        x=layers.Activation("relu")(x); x=layers.MaxPooling2D()(x); x=layers.Dropout(.2)(x)
    x=layers.Conv2D(256,3,padding="same",activation="relu")(x); x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dense(128,activation="relu")(x); x=layers.Dropout(.4)(x)
    out=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    m=keras.Model(inp,out,name="custom_cnn")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return m
custom_model=build_custom_cnn()
history_custom=custom_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,
    callbacks=callbacks("custom_cnn"),class_weight=class_weights)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

- Model: A custom CNN (Convolutional Neural Network) with convolution, batch normalization, pooling, dropout, and softmax layers was used for brain tumor classification.
- Performance: The model performance is evaluated using Accuracy, Precision, Recall, F1-Score, and Confusion Matrix to measure classification effectiveness and errors.

In [ ]:
# Visualizing evaluation Metric Score chart
def plot_history(history,title):
    h=history.history; fig,ax=plt.subplots(1,2,figsize=(11,4))
    ax[0].plot(h["accuracy"],label="train"); ax[0].plot(h["val_accuracy"],label="validation"); ax[0].set_title(title+" accuracy"); ax[0].legend()
    ax[1].plot(h["loss"],label="train"); ax[1].plot(h["val_loss"],label="validation"); ax[1].set_title(title+" loss"); ax[1].legend()
    plt.tight_layout(); plt.show()
plot_history(history_custom,"Custom CNN")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Simple controlled tuning experiment: lower learning rate and stronger dropout for custom CNN.
def build_tuned_cnn():
    inp=keras.Input((*IMG_SIZE,3)); x=layers.Rescaling(1./255)(inp); x=augmentation(x)
    for f in [32,64,128]:
        x=layers.Conv2D(f,3,padding="same",activation="relu")(x); x=layers.BatchNormalization()(x)
        x=layers.MaxPooling2D()(x); x=layers.Dropout(.3)(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dense(128,activation="relu")(x); x=layers.Dropout(.5)(x)
    out=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    m=keras.Model(inp,out,name="custom_cnn_tuned")
    m.compile(optimizer=keras.optimizers.Adam(3e-4),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return m
tuned_custom=build_tuned_cnn()
history_tuned=tuned_custom.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,
    callbacks=callbacks("custom_cnn_tuned"),class_weight=class_weights)


##### Which hyperparameter optimization technique have you used and why?

- Manual/controlled hyperparameter tuning was used.
- Learning rate was reduced from 1e-3 to 3e-4 and dropout was increased to reduce overfitting and improve validation performance.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

- Improvement is determined by comparing Accuracy, Precision, Recall, and F1-Score of the original and tuned CNN.
- If the tuned model shows higher validation/test scores and lower validation loss, it indicates better performance and generalization.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

- Model: The Tuned Custom CNN uses 3 convolutional blocks, batch normalization, dropout, and a softmax output for brain tumor classification.
- Performance: The training history chart compares training and validation accuracy/loss, helping assess model learning, convergence, and overfitting.

In [ ]:
# Visualizing evaluation Metric Score chart
plot_history(history_tuned,"Tuned Custom CNN")


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
def build_mobilenet():
    base=MobileNetV2(include_top=False,weights="imagenet",input_shape=(*IMG_SIZE,3)); base.trainable=False
    inp=keras.Input((*IMG_SIZE,3)); x=augmentation(inp); x=mobilenet_preprocess(x); x=base(x,training=False)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(.35)(x)
    out=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    m=keras.Model(inp,out,name="mobilenetv2")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return m
mobilenet_model=build_mobilenet()
history_mobilenet=mobilenet_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,
    callbacks=callbacks("mobilenetv2"),class_weight=class_weights)


##### Which hyperparameter optimization technique have you used and why?

- Transfer learning with a pre-trained MobileNetV2 was used, with the base layers frozen and Adam optimizer (learning rate = 0.001).
- This approach was chosen to reduce training time and use features already learned from ImageNet.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

- Improvement should be checked by comparing Accuracy, Precision, Recall, and F1-Score of MobileNetV2 with the previous CNN models.
- Higher validation/test scores and lower loss indicate better performance and generalization.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

- Accuracy: Overall correct predictions → improves confidence in the automated tumor classification system.
- Precision: Correctness of positive tumor predictions → reduces false alarms and unnecessary follow-up.
- Recall: Ability to detect actual tumor cases → important for reducing missed tumor cases.
- F1-Score: Balance between precision and recall → provides a reliable measure for overall model effectiveness.
- Business Impact: A well-performing model can support faster, consistent, and cost-effective medical image analysis, while poor recall can create a negative impact due to missed tumor detections.

### ML Model - 3

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnet():
    base=EfficientNetB0(include_top=False,weights="imagenet",input_shape=(*IMG_SIZE,3)); base.trainable=False
    inp=keras.Input((*IMG_SIZE,3)); x=augmentation(inp); x=base(x,training=False)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(.35)(x)
    out=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    m=keras.Model(inp,out,name="efficientnetb0")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return m
efficient_model=build_efficientnet()
history_efficient=efficient_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,
    callbacks=callbacks("efficientnetb0"),class_weight=class_weights)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

- Model: EfficientNetB0 with ImageNet pre-trained weights was used through transfer learning for brain tumor classification. The base layers were frozen and a custom classification layer was added.
- Performance: The model is evaluated using Accuracy, Precision, Recall, and F1-Score; higher scores indicate better classification performance and reliable generalization.

In [ ]:
# Visualizing evaluation Metric Score chart
plot_history(history_mobilenet,"MobileNetV2")
plot_history(history_efficient,"EfficientNetB0")


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
def evaluate_model(model,ds,name):
    y_true=np.concatenate([y.numpy() for _,y in ds])
    probs=model.predict(ds,verbose=0); y_pred=np.argmax(probs,axis=1)
    acc=accuracy_score(y_true,y_pred)
    pr,re,f1,_=precision_recall_fscore_support(y_true,y_pred,average="weighted",zero_division=0)
    print(f"{name}: accuracy={acc:.4f}, precision={pr:.4f}, recall={re:.4f}, weighted F1={f1:.4f}")
    print(classification_report(y_true,y_pred,target_names=classes,zero_division=0))
    cm=confusion_matrix(y_true,y_pred,labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(7,6)); sns.heatmap(cm,annot=True,fmt="d",xticklabels=classes,yticklabels=classes,cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(name+" confusion matrix"); plt.tight_layout(); plt.show()
    return {"model":name,"accuracy":acc,"precision":pr,"recall":re,"f1":f1}
# Evaluate on test only after training/model decisions are complete.
results=[
 evaluate_model(custom_model,test_ds,"Custom CNN"),
 evaluate_model(tuned_custom,test_ds,"Tuned Custom CNN"),
 evaluate_model(mobilenet_model,test_ds,"MobileNetV2"),
 evaluate_model(efficient_model,test_ds,"EfficientNetB0")]
results_df=pd.DataFrame(results).sort_values("f1",ascending=False)
display(results_df)


##### Which hyperparameter optimization technique have you used and why?

- Manual/controlled hyperparameter tuning was used by changing the learning rate and dropout rate.
- It was used to reduce overfitting and improve model generalization.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

- Improvement is identified by comparing the Accuracy, Precision, Recall, and F1-Score of all models.
- The model with the highest F1-Score and strong accuracy/recall is considered better.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

- Accuracy: Measures overall correct predictions.
- Precision: Helps reduce false tumor predictions.
- Recall: Important for detecting actual tumor cases and reducing missed cases.
- F1-Score: Balances precision and recall, making it a strong metric for medical classification

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

- The model with the highest weighted F1-Score on the test dataset should be selected as the final model.
- It provides the best balance between precision and recall and is more reliable for tumor classification.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

- CNN/transfer-learning model: Learns important visual patterns such as edges, textures, shapes, and tumor regions from MRI images.
- Grad-CAM can be used for explainability to highlight the MRI regions that influenced the model's prediction.
- This improves interpretability and trust by showing whether the model focuses on relevant tumor regions.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save selected checkpoint/model and class labels for inference.
best_name=results_df.iloc[0]["model"]
model_map={"Custom CNN":custom_model,"Tuned Custom CNN":tuned_custom,
           "MobileNetV2":mobilenet_model,"EfficientNetB0":efficient_model}
best_model=model_map[best_name]
best_model.save(os.path.join(OUT_DIR,"best_brain_tumor_model.keras"))
with open(os.path.join(OUT_DIR,"class_names.json"),"w") as f: json.dump(classes,f)
results_df.to_csv(os.path.join(OUT_DIR,"model_comparison.csv"),index=False)
print("Saved model:",best_name,"to",OUT_DIR)


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Reload saved model and predict one held-out test image for pipeline sanity check.
saved_model=keras.models.load_model(os.path.join(OUT_DIR,"best_brain_tumor_model.keras"))
test_files=image_files(TEST_DIR)
if test_files:
    path=str(test_files[0])
    im=keras.utils.load_img(path,target_size=IMG_SIZE)
    x=keras.utils.img_to_array(im)[None,...]
    probs=saved_model.predict(x,verbose=0)[0]
    pred=int(np.argmax(probs))
    print("Image:",path)
    print("Predicted dataset label:",classes[pred])
    print("Score:",float(probs[pred]))
else:
    print("No test images found.")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

1. End-to-End AI Pipeline

MRI Image → Data Validation → Preprocessing → Augmentation → Feature Learning → Classification → Evaluation → Explainability

2. Automated Feature Extraction

The CNN automatically learns multi-level visual features such as edges → textures → shapes → high-level tumor patterns, eliminating manual feature engineering.

3. Four ML/DL Models Compared

The project provides a meaningful comparison between:
Custom CNN,
Tuned Custom CNN,
MobileNetV2,
EfficientNetB0

4. Imbalance-Aware Learning

Instead of simply deleting data or oversampling, class weighting was used so that minority tumor classes receive greater importance during model training.

5. Data Augmentation

Rotation, zoom, translation, and horizontal flipping were applied to increase image variability and improve the model's ability to generalize to unseen MRI images.

6. Hyperparameter Optimization

The custom CNN was improved through controlled tuning of learning rate and dropout, targeting better generalization and reduced overfitting.

7. Transfer Learning

MobileNetV2 and EfficientNetB0 with ImageNet pretrained weights were used to leverage powerful previously learned visual representations.

8. Strong Evaluation Strategy

The models were evaluated on the test dataset only after training and model decisions were completed, using:
Accuracy + Precision + Recall + Weighted F1-Score + Confusion Matrix.

9. Medical AI Requires More Than Accuracy

For brain tumor classification, Recall and F1-Score are particularly important, because a model that achieves high accuracy but misses actual tumor cases can still be unsafe.

10. Final Takeaway

The major strength of this project is not simply building a CNN; it is the complete ML lifecycle—from data quality analysis and imbalance handling to model comparison, controlled optimization, unbiased test evaluation, and explainability.


“The proposed framework demonstrates how scalable deep-learning and transfer-learning techniques can automatically extract discriminative features from MRI images and classify brain tumor categories, while systematic evaluation and explainable AI improve the reliability, interpretability, and practical value of the model.”

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***